# ⭐ Day 28: Debugging in Python & Notebooks
## Mastering Error Diagnosis for AI & ML Development
### Day 28 of 369 • Python & AI Learning Path 🚀


## Introduction

Welcome to Day 28 of your 369-day Python & AI Learning Path! Today we tackle one of the most critical yet often overlooked skills in AI and ML development: **debugging**. While tutorials and courses focus heavily on model architectures and algorithms, the reality of production ML work is that you'll spend a significant portion of your time hunting down bugs, understanding mysterious failures, and tracing data through complex pipelines.

Here's the truth that separates beginners from professionals: **the best ML engineers aren't those who write perfect code on the first try—they're the ones who can diagnose and fix problems systematically.** When you're training a model for 12 hours only to discover NaN losses at epoch 47, or when your inference pipeline silently produces incorrect predictions for edge cases, strong debugging skills save you days of frustration and prevent costly mistakes in production systems.

AI and ML development presents unique debugging challenges. Unlike traditional software where errors are often immediate and obvious, ML bugs can be **silent**, **stochastic**, or **delayed**. A data preprocessing step might produce subtly wrong tensors that flow through your entire pipeline. A shape mismatch might only appear on specific batch sizes. Memory leaks might accumulate over thousands of iterations. These aren't just coding errors—they're complex system behaviors that require methodical investigation.

Today's notebook is designed as a **comprehensive debugging reference guide**. We'll move beyond simple print statements to master professional debugging tools, develop systematic workflows for diagnosing ML-specific issues, and build preventive practices that catch errors before they propagate. By the end of this session, you'll have the confidence to tackle even the most perplexing bugs in your AI projects. Let's turn debugging from a source of frustration into your professional superpower! 💪


## 📋 Table of Contents

1. [Common Python Errors in ML Context](#common-errors)
2. [Strategic Use of print() and pprint()](#print-debugging)
3. [Assert Statements for Early Detection](#assertions)
4. [Python Debugger (pdb)](#pdb-debugging)
5. [Jupyter Debugging Tools](#jupyter-tools)
6. [Understanding Tracebacks](#tracebacks)
7. [ML-Specific Bugs & Fixes](#ml-bugs)
8. [Systematic Debugging Workflow](#workflow)
9. [Preventive Practices](#prevention)
10. [🛠️ Hands-On Exercises](#exercises)
11. [Solutions](#solutions)


<a id='common-errors'></a>
## 🔍 Common Python Errors in ML Context

Understanding error types is the first step to rapid diagnosis. Here are the most common culprits in ML code:


In [1]:
# ⚠️ BUG 1: NameError - Using undefined variables
# Common when refactoring or copying code between notebooks

def preprocess_data(data):
    # Oops! 'scaler' was never defined
    return scaler.fit_transform(data)

# Uncomment to see the error:
# preprocess_data([[1, 2], [3, 4]])

# 🔧 FIX: Define the variable before use
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
preprocess_data([[1, 2], [3, 4]])

array([[-1., -1.],
       [ 1.,  1.]])

In [2]:
# ⚠️ BUG 2: TypeError - Operations on incompatible types
# Very common when mixing lists, arrays, and tensors

import numpy as np

predictions = np.array([0.1, 0.9, 0.3])
labels = [0, 1, 0]  # List instead of array!

# This works, but this won't:
# result = predictions + labels  # Actually works due to broadcasting

# Real bug: Trying to use list methods on arrays
# predictions.append(0.5)  # AttributeError - numpy arrays use np.append()

# 🔧 FIX: Ensure consistent types
labels_array = np.array(labels)
print(f"Type consistency check: {type(predictions)} + {type(labels_array)}")

Type consistency check: <class 'numpy.ndarray'> + <class 'numpy.ndarray'>


In [3]:
# ⚠️ BUG 3: ValueError - Correct type but invalid value
# Classic ML bug: shape mismatches

import numpy as np

weights = np.random.randn(10, 5)  # Expects 10 features
sample = np.random.randn(8)       # Only 8 features provided!

# Uncomment to see the error:
# output = np.dot(weights, sample)

# 🔧 FIX: Validate shapes before operations
print(f"Weights shape: {weights.shape}")
print(f"Sample shape: {sample.shape}")

if weights.shape[0] != sample.shape[0]:
    print(f"❌ Shape mismatch! Expected {weights.shape[0]}, got {sample.shape[0]}")
    sample = np.pad(sample, (0, weights.shape[0] - sample.shape[0]))
    print(f"✅ Padded sample to shape: {sample.shape}")

output = np.dot(weights.T, sample)  # Note: weights.T for proper matmul
print(f"Output shape: {output.shape}")

Weights shape: (10, 5)
Sample shape: (8,)
❌ Shape mismatch! Expected 10, got 8
✅ Padded sample to shape: (10,)
Output shape: (5,)


In [4]:
# ⚠️ BUG 4: IndexError and KeyError - Accessing non-existent indices/keys
# Common in data processing pipelines

batch_data = {
    'features': np.random.randn(32, 10),
    'labels': np.random.randint(0, 2, 32)
}

# KeyError: Trying to access 'target' instead of 'labels'
# targets = batch_data['target']  # Wrong key!

# IndexError: Accessing batch index beyond size
# single_example = batch_data['features'][50]  # Batch only has 32 items!

# 🔧 FIX: Use .get() for dictionaries and check lengths
targets = batch_data.get('labels')  # Safe access
print(f"Available keys: {list(batch_data.keys())}")

batch_size = len(batch_data['features'])
idx = 50
if idx < batch_size:
    single_example = batch_data['features'][idx]
else:
    print(f"⚠️ Index {idx} out of bounds for batch size {batch_size}")
    single_example = batch_data['features'][batch_size - 1]

Available keys: ['features', 'labels']
⚠️ Index 50 out of bounds for batch size 32


In [5]:
# ⚠️ BUG 5: AttributeError - Object doesn't have expected method/attribute
# Common when mixing pandas DataFrames with numpy arrays

import pandas as pd

df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
array = df.values  # Convert to numpy array

# This works on DataFrame but not array:
# array.columns  # AttributeError!

# 🔧 FIX: Check object type before accessing attributes
print(f"Type: {type(array)}")
if hasattr(array, 'columns'):
    print(f"Columns: {array.columns}")
else:
    print(f"Shape: {array.shape} (use .shape for arrays)")

# Safe method access with getattr
shape_info = getattr(array, 'shape', 'No shape attribute')
print(f"Shape info: {shape_info}")

Type: <class 'numpy.ndarray'>
Shape: (3, 2) (use .shape for arrays)
Shape info: (3, 2)


In [6]:
# ⚠️ BUG 6: ImportError/ModuleNotFoundError - Missing dependencies
# Common in new environments or when sharing code

# Uncomment to see error:
# import nonexistent_package

# 🔧 FIX: Check imports with try/except and helpful messages
try:
    import torch
    print(f"✅ PyTorch version: {torch.__version__}")
except ImportError:
    print("❌ PyTorch not installed. Run: pip install torch")
    
try:
    import sklearn
    print(f"✅ Scikit-learn version: {sklearn.__version__}")
except ImportError:
    print("❌ Scikit-learn not installed. Run: pip install scikit-learn")

❌ PyTorch not installed. Run: pip install torch
✅ Scikit-learn version: 1.6.1


<a id='print-debugging'></a>
## 🖨️ Strategic Use of print() and pprint()

Sometimes the simplest tools are the most effective. Strategic printing helps you understand data flow and state.


In [7]:
# 💡 Advanced printing techniques for ML debugging

import numpy as np
from pprint import pprint

# Create a complex nested structure (common in ML configs)
config = {
    'model': {
        'type': 'Transformer',
        'layers': 12,
        'heads': 8,
        'hidden_dim': 768
    },
    'training': {
        'batch_size': 32,
        'learning_rate': 0.0001,
        'optimizer': 'AdamW',
        'scheduler': {
            'type': 'cosine',
            'warmup_steps': 1000
        }
    },
    'data': {
        'path': '/datasets/imdb',
        'max_length': 512
    }
}

# Standard print - hard to read for nested structures
print("Standard print:")
print(config)
print()

# pprint - much better for nested dicts
print("Pretty print:")
pprint(config, depth=2, compact=True)

Standard print:
{'model': {'type': 'Transformer', 'layers': 12, 'heads': 8, 'hidden_dim': 768}, 'training': {'batch_size': 32, 'learning_rate': 0.0001, 'optimizer': 'AdamW', 'scheduler': {'type': 'cosine', 'warmup_steps': 1000}}, 'data': {'path': '/datasets/imdb', 'max_length': 512}}

Pretty print:
{'data': {'max_length': 512, 'path': '/datasets/imdb'},
 'model': {'heads': 8, 'hidden_dim': 768, 'layers': 12, 'type': 'Transformer'},
 'training': {'batch_size': 32,
              'learning_rate': 0.0001,
              'optimizer': 'AdamW',
              'scheduler': {...}}}


In [8]:
# 💡 Strategic debug printing for tensor operations

def debug_tensor_operation(tensor, operation_name="Operation"):
    """Print comprehensive debug info about a tensor/array."""
    print(f"\n🔍 DEBUG: {operation_name}")
    print(f"   Shape: {tensor.shape}")
    print(f"   Dtype: {tensor.dtype}")
    print(f"   Range: [{tensor.min():.4f}, {tensor.max():.4f}]")
    print(f"   Mean: {tensor.mean():.4f}, Std: {tensor.std():.4f}")
    print(f"   NaN count: {np.isnan(tensor).sum()}")
    print(f"   Inf count: {np.isinf(tensor).sum()}")
    return tensor

# Simulate a neural network layer with debug info
np.random.seed(42)
x = np.random.randn(4, 10)
x = debug_tensor_operation(x, "Input")

w = np.random.randn(10, 5) * 0.1
z = np.dot(x, w)
z = debug_tensor_operation(z, "After Linear Layer")

# Simulate activation
a = np.maximum(z, 0)  # ReLU
a = debug_tensor_operation(a, "After ReLU")


🔍 DEBUG: Input
   Shape: (4, 10)
   Dtype: float64
   Range: [-1.9597, 1.8523]
   Mean: -0.2186, Std: 0.9408
   NaN count: 0
   Inf count: 0

🔍 DEBUG: After Linear Layer
   Shape: (4, 5)
   Dtype: float64
   Range: [-0.6159, 0.3396]
   Mean: -0.0312, Std: 0.2115
   NaN count: 0
   Inf count: 0

🔍 DEBUG: After ReLU
   Shape: (4, 5)
   Dtype: float64
   Range: [0.0000, 0.3396]
   Mean: 0.0640, Std: 0.1067
   NaN count: 0
   Inf count: 0


In [9]:
# 💡 Using f-strings for formatted debug output

def train_step_debug(batch_idx, loss, metrics, epoch=0):
    """Formatted debug output for training loops."""
    print(
        f"Epoch {epoch:2d} | "
        f"Batch {batch_idx:4d} | "
        f"Loss: {loss:8.4f} | "
        f"Acc: {metrics['accuracy']:.2%} | "
        f"LR: {metrics['lr']:.2e}"
    )

# Simulate training output
import random
for epoch in range(2):
    for batch in range(5):
        loss = 2.0 * np.exp(-epoch * 0.5) + random.random() * 0.1
        acc = 0.5 + (epoch * 5 + batch) * 0.05
        metrics = {'accuracy': min(acc, 0.99), 'lr': 0.001}
        train_step_debug(batch, loss, metrics, epoch)

Epoch  0 | Batch    0 | Loss:   2.0342 | Acc: 50.00% | LR: 1.00e-03
Epoch  0 | Batch    1 | Loss:   2.0163 | Acc: 55.00% | LR: 1.00e-03
Epoch  0 | Batch    2 | Loss:   2.0766 | Acc: 60.00% | LR: 1.00e-03
Epoch  0 | Batch    3 | Loss:   2.0030 | Acc: 65.00% | LR: 1.00e-03
Epoch  0 | Batch    4 | Loss:   2.0119 | Acc: 70.00% | LR: 1.00e-03
Epoch  1 | Batch    0 | Loss:   1.2180 | Acc: 75.00% | LR: 1.00e-03
Epoch  1 | Batch    1 | Loss:   1.3119 | Acc: 80.00% | LR: 1.00e-03
Epoch  1 | Batch    2 | Loss:   1.2502 | Acc: 85.00% | LR: 1.00e-03
Epoch  1 | Batch    3 | Loss:   1.2153 | Acc: 90.00% | LR: 1.00e-03
Epoch  1 | Batch    4 | Loss:   1.2771 | Acc: 95.00% | LR: 1.00e-03


<a id='assertions'></a>
## ✅ Assert Statements for Early Error Detection

Assertions are your first line of defense. They catch errors immediately rather than letting them propagate silently.


In [10]:
# 💡 Using assertions to catch shape mismatches immediately

import numpy as np

def matrix_multiply_safe(a, b):
    """Matrix multiplication with assertion checks."""
    
    # Check inputs are arrays
    assert isinstance(a, np.ndarray), f"Expected ndarray, got {type(a)}"
    assert isinstance(b, np.ndarray), f"Expected ndarray, got {type(b)}"
    
    # Check dimensions
    assert a.ndim == 2, f"Matrix a must be 2D, got shape {a.shape}"
    assert b.ndim == 2, f"Matrix b must be 2D, got shape {b.shape}"
    
    # Check compatibility
    assert a.shape[1] == b.shape[0], \
        f"Shape mismatch: {a.shape} @ {b.shape} - inner dimensions must match"
    
    result = np.dot(a, b)
    
    # Verify output shape
    assert result.shape == (a.shape[0], b.shape[1]), "Output shape incorrect"
    
    return result

# Test with valid input
a = np.random.randn(3, 4)
b = np.random.randn(4, 5)
result = matrix_multiply_safe(a, b)
print(f"✅ Success: {a.shape} @ {b.shape} = {result.shape}")

# Test with invalid input (uncomment to see assertion error):
# c = np.random.randn(4, 5)
# matrix_multiply_safe(a, c)  # AssertionError: inner dimensions don't match

✅ Success: (3, 4) @ (4, 5) = (3, 5)


In [11]:
# 💡 Assertions for data validation in ML pipelines

def preprocess_batch(images, labels, expected_shape=(32, 224, 224, 3)):
    """Preprocess with comprehensive validation assertions."""
    
    # Type checks
    assert isinstance(images, np.ndarray), "Images must be numpy array"
    assert images.dtype in [np.float32, np.float64], \
        f"Images should be float, got {images.dtype}"
    
    # Shape checks
    assert images.ndim == 4, f"Expected 4D batch (B,H,W,C), got {images.ndim}D"
    assert images.shape[0] == labels.shape[0], "Batch size mismatch between images and labels"
    
    # Value checks
    assert not np.isnan(images).any(), "Images contain NaN values!"
    assert not np.isinf(images).any(), "Images contain Inf values!"
    assert images.min() >= 0 and images.max() <= 255, \
        f"Pixel values out of range [0, 255]: [{images.min()}, {images.max()}]"
    
    # Range normalization
    normalized = images / 255.0
    
    assert normalized.min() >= 0 and normalized.max() <= 1, "Normalization failed"
    
    return normalized, labels

# Create valid test data
test_images = np.random.randint(0, 256, (16, 224, 224, 3)).astype(np.float32)
test_labels = np.random.randint(0, 10, 16)

processed, labels = preprocess_batch(test_images, test_labels)
print(f"✅ Batch processed successfully: shape={processed.shape}, range=[{processed.min():.2f}, {processed.max():.2f}]")

✅ Batch processed successfully: shape=(16, 224, 224, 3), range=[0.00, 1.00]


In [12]:
# 💡 Assertions with helpful error messages for debugging

def compute_loss(predictions, targets, epsilon=1e-7):
    """Compute cross-entropy loss with validation."""
    
    # Shape validation with detailed messages
    assert predictions.shape == targets.shape, \
        f"Shape mismatch: predictions {predictions.shape} vs targets {targets.shape}"
    
    # Probability validation (predictions should be probabilities)
    assert (predictions >= 0).all() and (predictions <= 1).all(), \
        f"Predictions must be in [0, 1], got range [{predictions.min()}, {predictions.max()}]"
    
    # Check for valid probability distributions (sum to ~1 for multi-class)
    if predictions.ndim > 1 and predictions.shape[-1] > 1:
        row_sums = predictions.sum(axis=-1)
        assert np.allclose(row_sums, 1.0, atol=0.01), \
            f"Predictions don't sum to 1 (row sums: {row_sums[:5]})"
    
    # Numerical stability check
    clipped_preds = np.clip(predictions, epsilon, 1 - epsilon)
    
    loss = -np.sum(targets * np.log(clipped_preds)) / len(targets)
    
    assert not np.isnan(loss), "Loss is NaN - check input values"
    assert not np.isinf(loss), "Loss is Inf - numerical instability detected"
    
    return loss

# Valid case
valid_preds = np.array([[0.9, 0.1], [0.2, 0.8]])
valid_targets = np.array([[1, 0], [0, 1]])
loss = compute_loss(valid_preds, valid_targets)
print(f"✅ Valid loss computed: {loss:.4f}")

# Invalid case (uncomment to see assertion):
# invalid_preds = np.array([[0.5, 0.3], [0.2, 0.8]])  # Doesn't sum to 1
# compute_loss(invalid_preds, valid_targets)

✅ Valid loss computed: 0.1643


<a id='pdb-debugging'></a>
## 🐛 Python Debugger (pdb)

When print statements aren't enough, the Python Debugger (pdb) lets you inspect running code interactively.


In [13]:
# 💡 Basic pdb commands demonstration
# Commands: n (next), c (continue), s (step), p (print), q (quit)

import pdb
import numpy as np

def buggy_neural_layer(x, w, b):
    """A layer with a subtle bug for debugging practice."""
    print("Starting forward pass...")
    
    # Set a breakpoint here to inspect values
    # pdb.set_trace()  # Uncomment to enter debugger
    
    z = np.dot(x, w) + b
    
    # Bug: ReLU implemented incorrectly (should be max(0, z))
    # This version zeros out positive values instead of negative ones
    a = np.where(z > 0, 0, z)  # Bug: inverted condition!
    
    return a

# Setup test data
np.random.seed(42)
x = np.random.randn(2, 3)
w = np.random.randn(3, 4)
b = np.random.randn(4)

print("Input x:", x)
print("Weights w:", w)

# Run the function
output = buggy_neural_layer(x, w, b)
print("Output:", output)
print("⚠️ Notice all positive values are zeroed - that's the bug!")

# 💡 DEBUGGING WORKFLOW:
# 1. Uncomment pdb.set_trace() above
# 2. Run the cell - execution will pause at the breakpoint
# 3. In the prompt, try these commands:
#    p z        - print the z values
#    p z.shape  - check shape
#    p z > 0    - see the boolean mask
#    n          - execute next line
#    p a        - see the buggy output
#    q          - quit debugger

Input x: [[ 0.49671415 -0.1382643   0.64768854]
 [ 1.52302986 -0.23415337 -0.23413696]]
Weights w: [[ 1.57921282  0.76743473 -0.46947439  0.54256004]
 [-0.46341769 -0.46572975  0.24196227 -1.91328024]
 [-1.72491783 -0.56228753 -1.01283112  0.31424733]]
Starting forward pass...
Output: [[-1.17674211 -1.3309014   0.          0.        ]
 [ 0.         -0.00277321  0.          0.        ]]
⚠️ Notice all positive values are zeroed - that's the bug!


In [14]:
# 💡 Post-mortem debugging with pdb.pm()
# This enters the debugger at the point where the last exception occurred

import pdb
import sys

def failing_function():
    """A function that will raise an exception."""
    data = {'values': [1, 2, 3]}
    
    # This will raise KeyError
    result = data['valuse']  # Typo: 'valuse' instead of 'values'
    return result

# Uncomment to trigger exception and enter post-mortem debugging:
# try:
#     failing_function()
# except Exception:
#     pdb.pm()  # Post-mortem - drops you into the stack frame where exception occurred

# In pdb.pm() prompt, you can:
# p data       - see what keys are actually available
# p data.keys() - list all keys
# p data['values'] - access the correct key
# q            - quit

print("💡 Tip: After an exception, run pdb.pm() to inspect the crash site")

💡 Tip: After an exception, run pdb.pm() to inspect the crash site


In [15]:
# 💡 Using pdb in training loops for conditional debugging

import numpy as np

def train_with_debug(model_data, epochs=3):
    """Training loop with conditional breakpoint."""
    losses = []
    
    for epoch in range(epochs):
        for batch_idx in range(5):
            # Simulate loss calculation
            loss = 1.0 / (epoch + batch_idx + 1)
            
            # Conditional breakpoint: stop if loss is NaN
            if np.isnan(loss):
                print(f"NaN detected at epoch {epoch}, batch {batch_idx}")
                import pdb; pdb.set_trace()
            
            losses.append(loss)
            
            # Another condition: stop for inspection every 3rd batch
            if batch_idx == 2:
                print(f"Breakpoint batch - Epoch {epoch}, Loss: {loss:.4f}")
                # Uncomment to inspect:
                # import pdb; pdb.set_trace()
    
    return losses

losses = train_with_debug(None)
print(f"Training completed. Final loss: {losses[-1]:.4f}")

Breakpoint batch - Epoch 0, Loss: 0.3333
Breakpoint batch - Epoch 1, Loss: 0.2500
Breakpoint batch - Epoch 2, Loss: 0.2000
Training completed. Final loss: 0.1429


<a id='jupyter-tools'></a>
## 🔬 Jupyter Debugging Tools

Jupyter provides magic commands that make debugging even more convenient.


In [16]:
# 💡 %pdb magic - automatically enter debugger on exceptions

# Uncomment to enable automatic debugging:
# %pdb on

def function_that_fails():
    x = np.array([1, 2, 3])
    y = np.array([1, 2])  # Different shape!
    return x + y  # This will raise ValueError

# With %pdb on, this will drop you into the debugger at the exception point:
# function_that_fails()

print("Enable %pdb on to automatically enter debugger when exceptions occur")

Enable %pdb on to automatically enter debugger when exceptions occur


In [17]:
# 💡 %debug magic - post-mortem debugging after an exception

def calculate_metrics(predictions, targets):
    """Calculate accuracy with potential bugs."""
    correct = (predictions == targets).sum()
    total = len(targets)
    
    # Potential division by zero if targets is empty
    accuracy = correct / total
    
    return accuracy

# First, trigger an exception:
try:
    acc = calculate_metrics(np.array([1, 2]), np.array([]))  # Empty targets!
except Exception as e:
    print(f"Exception caught: {e}")
    print("Now run %debug in the next cell to inspect!")

# After running the above cell and getting an error, 
# run %debug in a new cell to enter the debugger at the exception point

Exception caught: operands could not be broadcast together with shapes (2,) (0,) 
Now run %debug in the next cell to inspect!


In [18]:
# 💡 %xmode magic - control exception reporting verbosity

# Available modes: Plain, Context, Verbose
# %xmode Verbose  # Most detailed - shows local variables at each frame

def nested_function():
    data = np.random.randn(3, 3)
    result = np.linalg.inv(data)
    return result

def outer_function():
    values = [1, 2, 3]
    return nested_function()

# Uncomment to see verbose traceback:
# outer_function()

print("Use %xmode Verbose for detailed tracebacks with local variables")

Use %xmode Verbose for detailed tracebacks with local variables


In [19]:
# 💡 Practical example: Debugging a shape mismatch with %debug

import numpy as np

def batch_predict(model_weights, batch_data):
    """Make predictions on a batch."""
    # Bug: We transpose weights when we shouldn't
    predictions = np.dot(batch_data, model_weights.T)  # Wrong!
    return predictions

# Setup
weights = np.random.randn(10, 5)  # 10 features -> 5 outputs
batch = np.random.randn(32, 10)   # 32 samples, 10 features

print(f"Weights shape: {weights.shape}")
print(f"Batch shape: {batch.shape}")
print(f"Expected output shape: (32, 5)")

try:
    preds = batch_predict(weights, batch)
    print(f"Actual output shape: {preds.shape}")
except Exception as e:
    print(f"Error: {e}")
    print("\n💡 DEBUGGING STEPS:")
    print("1. Run this cell to see the error")
    print("2. Run %debug in a new cell")
    print("3. In debugger, check: p model_weights.shape, p batch_data.shape")
    print("4. Try: p np.dot(batch_data, model_weights).shape (without .T)")
    print("5. Fix: Remove the .T transpose in the function")

Weights shape: (10, 5)
Batch shape: (32, 10)
Expected output shape: (32, 5)
Error: shapes (32,10) and (5,10) not aligned: 10 (dim 1) != 5 (dim 0)

💡 DEBUGGING STEPS:
1. Run this cell to see the error
2. Run %debug in a new cell
3. In debugger, check: p model_weights.shape, p batch_data.shape
4. Try: p np.dot(batch_data, model_weights).shape (without .T)
5. Fix: Remove the .T transpose in the function


<a id='tracebacks'></a>
## 📖 Reading and Understanding Tracebacks

A traceback is your roadmap to the bug. Learning to read it efficiently saves enormous time.


In [20]:
# 💡 Anatomy of a traceback demonstration

import numpy as np

def layer1(x):
    w = np.random.randn(10, 20)
    return np.dot(x, w)

def layer2(x):
    # Bug: wrong shape
    w = np.random.randn(15, 5)  # Expects 20 from previous layer!
    return np.dot(x, w)

def forward_pass(input_data):
    h1 = layer1(input_data)
    print(f"After layer1: {h1.shape}")
    h2 = layer2(h1)  # Error happens here
    return h2

# Trigger the error
input_data = np.random.randn(5, 10)

try:
    output = forward_pass(input_data)
except Exception as e:
    import traceback
    print("🔍 TRACEBACK ANALYSIS:")
    print("=" * 50)
    traceback.print_exc()
    print("=" * 50)
    print("\n📖 HOW TO READ THIS:")
    print("1. Start from the BOTTOM - that's where the actual error occurred")
    print("2. Read UPWARD to trace the call stack")
    print("3. Look for the line numbers and function names")
    print("4. The error message at the bottom tells you WHAT went wrong")
    print("5. The stack trace tells you WHERE it went wrong")
    print("\n💡 In this case:")
    print("   - ValueError: shapes (5,20) and (15,5) not aligned")
    print("   - layer1 outputs (5,20) but layer2 expects (20,?) or (?,5)")
    print("   - Fix: Change layer2 weights to shape (20, 5)")

After layer1: (5, 20)
🔍 TRACEBACK ANALYSIS:

📖 HOW TO READ THIS:
1. Start from the BOTTOM - that's where the actual error occurred
2. Read UPWARD to trace the call stack
3. Look for the line numbers and function names
4. The error message at the bottom tells you WHAT went wrong
5. The stack trace tells you WHERE it went wrong

💡 In this case:
   - ValueError: shapes (5,20) and (15,5) not aligned
   - layer1 outputs (5,20) but layer2 expects (20,?) or (?,5)
   - Fix: Change layer2 weights to shape (20, 5)


Traceback (most recent call last):
  File "C:\Users\786\AppData\Local\Temp\ipykernel_10364\1678098205.py", line 24, in <module>
    output = forward_pass(input_data)
  File "C:\Users\786\AppData\Local\Temp\ipykernel_10364\1678098205.py", line 17, in forward_pass
    h2 = layer2(h1)  # Error happens here
  File "C:\Users\786\AppData\Local\Temp\ipykernel_10364\1678098205.py", line 12, in layer2
    return np.dot(x, w)
           ~~~~~~^^^^^^
ValueError: shapes (5,20) and (15,5) not aligned: 20 (dim 1) != 15 (dim 0)


In [21]:
# 💡 Extracting useful information from tracebacks programmatically

import sys
import traceback

def robust_training_step(batch, model):
    """Training step with comprehensive error handling."""
    try:
        # Simulate processing
        if batch['data'].shape[0] == 0:
            raise ValueError("Empty batch received!")
        
        loss = model(batch)
        return loss
        
    except Exception as e:
        # Extract traceback information
        exc_type, exc_value, exc_traceback = sys.exc_info()
        
        print(f"❌ Error Type: {exc_type.__name__}")
        print(f"❌ Error Message: {exc_value}")
        print("\n📍 Stack Trace (most recent call last):")
        
        # Format traceback
        formatted = traceback.format_exception(exc_type, exc_value, exc_traceback)
        for line in formatted:
            print(line, end='')
        
        # Additional context
        print(f"\n🔍 Batch keys: {batch.keys()}")
        print(f"🔍 Data shape: {batch['data'].shape if 'data' in batch else 'N/A'}")
        
        raise  # Re-raise after logging

# Mock model
def mock_model(batch):
    return np.random.random()

# Test with invalid batch
empty_batch = {'data': np.array([]), 'labels': np.array([])}

try:
    robust_training_step(empty_batch, mock_model)
except:
    pass  # Expected to fail

❌ Error Type: ValueError
❌ Error Message: Empty batch received!

📍 Stack Trace (most recent call last):
Traceback (most recent call last):
  File "C:\Users\786\AppData\Local\Temp\ipykernel_10364\1689319246.py", line 11, in robust_training_step
    raise ValueError("Empty batch received!")
ValueError: Empty batch received!

🔍 Batch keys: dict_keys(['data', 'labels'])
🔍 Data shape: (0,)


<a id='ml-bugs'></a>
## 🧠 Common ML-Specific Bugs & Fixes

ML development has its own unique category of bugs. Let's explore the most common ones.


In [22]:
# ⚠️ BUG 1: Tensor/NumPy Shape Mismatches
# The most common ML error - dimension misalignment

import numpy as np

def matrix_multiply_buggy(a, b):
    """Common shape confusion in matrix operations."""
    # User expects (m,n) @ (n,p) = (m,p)
    # But might accidentally do (m,n) @ (p,n) by transposing wrong
    return np.dot(a, b.T)  # Bug: transposing wrong matrix!

# Correct usage
a = np.random.randn(5, 10)
b = np.random.randn(10, 3)

print(f"Matrix a: {a.shape}")
print(f"Matrix b: {b.shape}")
print(f"Expected: a @ b = (5, 3)")

try:
    result = matrix_multiply_buggy(a, b)
    print(f"Buggy result: {result.shape}")
except Exception as e:
    print(f"❌ Error: {e}")

# 🔧 FIX: Understand your dimensions
def matrix_multiply_correct(a, b):
    """Correct matrix multiplication with validation."""
    print(f"Multiplying {a.shape} @ {b.shape}")
    
    if a.shape[-1] != b.shape[0]:
        print(f"⚠️  Dimension mismatch! Trying b.T...")
        if a.shape[-1] == b.shape[-1]:
            b = b.T
        else:
            raise ValueError(f"Cannot multiply {a.shape} with {b.shape}")
    
    result = np.dot(a, b)
    print(f"✅ Result: {result.shape}")
    return result

correct_result = matrix_multiply_correct(a, b)

Matrix a: (5, 10)
Matrix b: (10, 3)
Expected: a @ b = (5, 3)
❌ Error: shapes (5,10) and (3,10) not aligned: 10 (dim 1) != 3 (dim 0)
Multiplying (5, 10) @ (10, 3)
✅ Result: (5, 3)


In [23]:
# ⚠️ BUG 2: Data Type Issues (object vs numeric)
# Silent killer - pandas DataFrames with object dtype

import pandas as pd
import numpy as np

# Create DataFrame with mixed types (common when loading CSV)
df = pd.DataFrame({
    'feature1': ['1.5', '2.5', '3.5', '4.5'],  # Strings!
    'feature2': [1, 2, 3, 4],
    'target': [0, 1, 0, 1]
})

print("DataFrame dtypes (notice object type!):")
print(df.dtypes)
print()

# Try to compute mean - this will fail or give wrong results
try:
    mean_val = df['feature1'].mean()
    print(f"Mean of feature1: {mean_val}")  # Wrong if it works!
except Exception as e:
    print(f"❌ Error computing mean: {e}")

# 🔧 FIX: Explicit type conversion
df['feature1'] = pd.to_numeric(df['feature1'], errors='coerce')
print(f"\n✅ After conversion: {df['feature1'].dtype}")
print(f"✅ Correct mean: {df['feature1'].mean():.2f}")

# Best practice: validate on load
def safe_load_data(filepath):
    """Load data with type validation."""
    df = pd.read_csv(filepath)
    
    # Check for object columns that should be numeric
    object_cols = df.select_dtypes(include=['object']).columns
    if len(object_cols) > 0:
        print(f"⚠️  Object columns detected: {list(object_cols)}")
        print("Attempting conversion...")
        for col in object_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df

print("\n💡 Best practice: Always check dtypes after loading data!")

DataFrame dtypes (notice object type!):
feature1      str
feature2    int64
target      int64
dtype: object

❌ Error computing mean: Cannot perform reduction 'mean' with string dtype

✅ After conversion: float64
✅ Correct mean: 3.00

💡 Best practice: Always check dtypes after loading data!


In [24]:
# ⚠️ BUG 3: NaN and Inf Propagation
# Silent failure that ruins model training

import numpy as np

# Simulate data with missing values
data = np.array([1.0, 2.0, np.nan, 4.0, 5.0, np.inf, 7.0])

def compute_statistics(values):
    """Compute stats that might hide NaN/Inf issues."""
    return {
        'mean': np.mean(values),
        'std': np.std(values),
        'min': np.min(values),
        'max': np.max(values)
    }

stats = compute_statistics(data)
print("Statistics (with NaN/Inf):")
for key, val in stats.items():
    print(f"  {key}: {val}")

# Notice: mean is NaN, max is Inf - but code didn't crash!

# 🔧 FIX: Detect and handle NaN/Inf early
def safe_compute_statistics(values):
    """Compute stats with NaN/Inf detection."""
    
    # Check for issues
    nan_count = np.isnan(values).sum()
    inf_count = np.isinf(values).sum()
    
    if nan_count > 0:
        print(f"⚠️  Detected {nan_count} NaN values")
        values = values[~np.isnan(values)]
    
    if inf_count > 0:
        print(f"⚠️  Detected {inf_count} Inf values")
        values = values[~np.isinf(values)]
    
    if len(values) == 0:
        raise ValueError("No valid values remaining after cleaning!")
    
    return {
        'mean': np.mean(values),
        'std': np.std(values),
        'min': np.min(values),
        'max': np.max(values),
        'valid_count': len(values)
    }

print("\nSafe statistics:")
safe_stats = safe_compute_statistics(data.copy())
for key, val in safe_stats.items():
    print(f"  {key}: {val:.4f}" if isinstance(val, float) else f"  {key}: {val}")

Statistics (with NaN/Inf):
  mean: nan
  std: nan
  min: nan
  max: nan

Safe statistics:
⚠️  Detected 1 NaN values
⚠️  Detected 1 Inf values
  mean: 3.8000
  std: 2.1354
  min: 1.0000
  max: 7.0000
  valid_count: 5


In [25]:
# ⚠️ BUG 4: Index Alignment Problems in Pandas
# Operations don't align as expected

import pandas as pd
import numpy as np

# Create two series with different indices
s1 = pd.Series([1, 2, 3, 4], index=['a', 'b', 'c', 'd'])
s2 = pd.Series([10, 20, 30], index=['b', 'c', 'e'])

print("Series 1:")
print(s1)
print("\nSeries 2:")
print(s2)

# Addition with different indices
result = s1 + s2
print("\nResult of s1 + s2:")
print(result)
print(f"\nNotice NaN where indices don't match!")

# 🔧 FIX: Align indices explicitly or reset them
# Option 1: Reset indices
s1_reset = s1.reset_index(drop=True)
s2_reset = s2.reset_index(drop=True)
print(f"\nAfter reset_index: {s1_reset[:3] + s2_reset[:3]}")

# Option 2: Align explicitly
s1_aligned, s2_aligned = s1.align(s2, join='inner')  # Only common indices
print(f"\nInner alignment result: {s1_aligned + s2_aligned}")

# Option 3: Reindex to match
s2_reindexed = s2.reindex(s1.index, fill_value=0)
print(f"\nReindexed addition: {(s1 + s2_reindexed).dropna()}")

Series 1:
a    1
b    2
c    3
d    4
dtype: int64

Series 2:
b    10
c    20
e    30
dtype: int64

Result of s1 + s2:
a     NaN
b    12.0
c    23.0
d     NaN
e     NaN
dtype: float64

Notice NaN where indices don't match!

After reset_index: 0    11
1    22
2    33
dtype: int64

Inner alignment result: b    12
c    23
dtype: int64

Reindexed addition: a     1
b    12
c    23
d     4
dtype: int64


In [26]:
# ⚠️ BUG 5: Memory and Performance-Related Errors
# Running out of memory during batch processing

import numpy as np
import sys

def process_large_dataset_buggy(data_list):
    """Process that accumulates all results in memory."""
    all_results = []
    
    for batch in data_list:
        # Process batch
        result = batch * 2 + 1
        all_results.append(result)  # Accumulates everything!
    
    return np.concatenate(all_results)  # Can cause MemoryError

def process_large_dataset_fixed(data_list, chunk_size=1000):
    """Memory-efficient processing with generators."""
    
    def process_generator():
        for i, batch in enumerate(data_list):
            result = batch * 2 + 1
            
            # Log progress for large datasets
            if i % 100 == 0:
                print(f"Processing batch {i}...")
            
            yield result
    
    # Process in chunks to control memory
    results = []
    for chunk in process_generator():
        results.append(chunk)
        
        # Clear periodically for very large datasets
        if len(results) >= chunk_size:
            partial = np.concatenate(results)
            print(f"Flushing {len(partial)} processed items")
            results = []  # Free memory
            yield partial
    
    if results:
        yield np.concatenate(results)

# Simulate large data
small_batches = [np.random.randn(100) for _ in range(10)]
result = process_large_dataset_buggy(small_batches)
print(f"Processed {len(result)} items")

print("\n💡 For truly large datasets, consider:")
print("   1. Using generators instead of lists")
print("   2. Processing in chunks")
print("   3. Using numpy memmap for disk-backed arrays")
print("   4. Dask for out-of-core computation")

Processed 1000 items

💡 For truly large datasets, consider:
   1. Using generators instead of lists
   2. Processing in chunks
   3. Using numpy memmap for disk-backed arrays
   4. Dask for out-of-core computation


In [27]:
# ⚠️ BUG 6: Silent Failures in Data Pipelines
# When errors are caught and ignored

import numpy as np
import warnings

def risky_preprocessing(data):
    """Preprocessing that might fail on edge cases."""
    if len(data) == 0:
        raise ValueError("Empty data!")
    
    # Normalize
    mean = np.mean(data)
    std = np.std(data)
    
    if std == 0:
        raise ValueError("Zero variance!")
    
    return (data - mean) / std

def buggy_pipeline(data_list):
    """Pipeline with silent failure handling."""
    results = []
    
    for i, data in enumerate(data_list):
        try:
            processed = risky_preprocessing(data)
            results.append(processed)
        except Exception as e:
            # ❌ BAD: Silently skipping failed items!
            pass  # Error ignored, data lost
    
    return results

def fixed_pipeline(data_list):
    """Pipeline with proper error handling."""
    results = []
    failed_indices = []
    
    for i, data in enumerate(data_list):
        try:
            processed = risky_preprocessing(data)
            results.append((i, processed))
        except Exception as e:
            # ✅ GOOD: Log the error and track failures
            warnings.warn(f"Item {i} failed: {e}")
            failed_indices.append(i)
            
            # Option: Use placeholder or interpolation
            results.append((i, np.zeros_like(data)))
    
    print(f"✅ Processed: {len(results) - len(failed_indices)}")
    print(f"⚠️  Failed: {len(failed_indices)} at indices {failed_indices}")
    
    return results

# Test data with edge cases
test_data = [
    np.array([1, 2, 3]),
    np.array([]),           # Empty - will fail
    np.array([5, 5, 5]),    # Zero variance - will fail
    np.array([4, 5, 6])
]

print("Fixed pipeline results:")
fixed_results = fixed_pipeline(test_data)
print(f"Total items returned: {len(fixed_results)}")

Fixed pipeline results:
✅ Processed: 2
⚠️  Failed: 2 at indices [1, 2]
Total items returned: 4


C:\Users\786\AppData\Local\Temp\ipykernel_10364\3948652884.py:46: UserWarning: Item 1 failed: Empty data!
  warnings.warn(f"Item {i} failed: {e}")
C:\Users\786\AppData\Local\Temp\ipykernel_10364\3948652884.py:46: UserWarning: Item 2 failed: Zero variance!
  warnings.warn(f"Item {i} failed: {e}")


<a id='workflow'></a>
## 🔄 Systematic Debugging Workflow for AI/ML Projects

When facing a mysterious bug, follow this systematic approach.


In [28]:
# 💡 Systematic debugging workflow demonstration

import numpy as np

class DebuggingWorkflow:
    """Demonstrates systematic debugging approach."""
    
    def __init__(self):
        self.debug_log = []
    
    def step1_reproduce(self, data):
        """STEP 1: Create minimal reproducible example."""
        print("🔍 STEP 1: Reproduce the bug")
        print("   - Isolate the failing case")
        print("   - Remove dependencies on external data")
        
        # Minimal example of the bug
        minimal_data = np.array([[1, 2], [3, 4]])
        self.debug_log.append("Reproduced with minimal data")
        return minimal_data
    
    def step2_inspect(self, data):
        """STEP 2: Inspect intermediate values."""
        print("\n🔍 STEP 2: Inspect state")
        
        print(f"   Shape: {data.shape}")
        print(f"   Dtype: {data.dtype}")
        print(f"   Range: [{data.min()}, {data.max()}]")
        print(f"   Sample: {data.flat[:5]}")
        
        self.debug_log.append(f"Inspected: shape={data.shape}")
        return data
    
    def step3_hypothesize(self, data):
        """STEP 3: Form hypothesis about cause."""
        print("\n🔍 STEP 3: Form hypothesis")
        print("   - What changed recently?")
        print("   - What assumptions am I making?")
        print("   - What does the error message actually say?")
        
        hypothesis = "Shape mismatch in matrix multiplication"
        print(f"   Hypothesis: {hypothesis}")
        self.debug_log.append(f"Hypothesis: {hypothesis}")
        return hypothesis
    
    def step4_test(self, data):
        """STEP 4: Test hypothesis with small experiment."""
        print("\n🔍 STEP 4: Test hypothesis")
        
        # Quick experiment
        try:
            result = np.dot(data, np.random.randn(3, 2))
            print("   ❌ Hypothesis wrong - no error with this shape")
        except ValueError as e:
            print(f"   ✅ Hypothesis confirmed: {e}")
        
        self.debug_log.append("Tested hypothesis")
    
    def step5_fix(self, data):
        """STEP 5: Implement and verify fix."""
        print("\n🔍 STEP 5: Implement fix")
        
        # Fix: Ensure compatible shapes
        fixed_data = data[:, :2]  # Adjust shape
        result = np.dot(fixed_data, np.random.randn(2, 3))
        print(f"   ✅ Fix verified: {fixed_data.shape} @ (2,3) = {result.shape}")
        
        self.debug_log.append("Fix implemented and verified")
    
    def run(self, data):
        """Execute full workflow."""
        print("=" * 60)
        print("SYSTEMATIC DEBUGGING WORKFLOW")
        print("=" * 60)
        
        data = self.step1_reproduce(data)
        data = self.step2_inspect(data)
        hypothesis = self.step3_hypothesize(data)
        self.step4_test(data)
        self.step5_fix(data)
        
        print("\n" + "=" * 60)
        print("DEBUG LOG:")
        for entry in self.debug_log:
            print(f"  • {entry}")
        print("=" * 60)

# Run the workflow demonstration
workflow = DebuggingWorkflow()
test_data = np.random.randn(100, 10)
workflow.run(test_data)

SYSTEMATIC DEBUGGING WORKFLOW
🔍 STEP 1: Reproduce the bug
   - Isolate the failing case
   - Remove dependencies on external data

🔍 STEP 2: Inspect state
   Shape: (2, 2)
   Dtype: int64
   Range: [1, 4]
   Sample: [1 2 3 4]

🔍 STEP 3: Form hypothesis
   - What changed recently?
   - What assumptions am I making?
   - What does the error message actually say?
   Hypothesis: Shape mismatch in matrix multiplication

🔍 STEP 4: Test hypothesis
   ✅ Hypothesis confirmed: shapes (2,2) and (3,2) not aligned: 2 (dim 1) != 3 (dim 0)

🔍 STEP 5: Implement fix
   ✅ Fix verified: (2, 2) @ (2,3) = (2, 3)

DEBUG LOG:
  • Reproduced with minimal data
  • Inspected: shape=(2, 2)
  • Hypothesis: Shape mismatch in matrix multiplication
  • Tested hypothesis
  • Fix implemented and verified


In [29]:
# 💡 Debugging checklist for ML projects

def ml_debugging_checklist():
    """Print a comprehensive debugging checklist."""
    checklist = """
    🐛 ML DEBUGGING CHECKLIST
    
    DATA VALIDATION:
    ☐ Check data shapes at each pipeline stage
    ☐ Verify dtypes (beware 'object' dtype!)
    ☐ Check for NaN/Inf values
    ☐ Validate index alignment in pandas
    ☐ Confirm train/validation/test splits are correct
    ☐ Check class balance in classification tasks
    
    MODEL ARCHITECTURE:
    ☐ Verify layer input/output shapes
    ☐ Check activation functions are appropriate
    ☐ Confirm loss function matches output type
    ☐ Validate optimizer parameters
    ☐ Check for gradient flow (vanishing/exploding)
    
    TRAINING LOOP:
    ☐ Monitor loss for NaN/Inf values
    ☐ Check learning rate (too high/low?)
    ☐ Verify batch size fits in memory
    ☐ Confirm data augmentation is working
    ☐ Check for data leakage between splits
    ☐ Validate metric calculations
    
    ENVIRONMENT:
    ☐ Confirm package versions match
    ☐ Check CUDA/GPU availability if expected
    ☐ Verify random seeds for reproducibility
    ☐ Check file paths and data loading
    ☐ Confirm sufficient disk/memory space
    
    REPRODUCIBILITY:
    ☐ Set all random seeds (numpy, random, torch, tensorflow)
    ☐ Log hyperparameters and config
    ☐ Save intermediate outputs for comparison
    ☐ Document environment setup
    """
    print(checklist)

ml_debugging_checklist()


    🐛 ML DEBUGGING CHECKLIST

    DATA VALIDATION:
    ☐ Check data shapes at each pipeline stage
    ☐ Verify dtypes (beware 'object' dtype!)
    ☐ Check for NaN/Inf values
    ☐ Validate index alignment in pandas
    ☐ Confirm train/validation/test splits are correct
    ☐ Check class balance in classification tasks

    MODEL ARCHITECTURE:
    ☐ Verify layer input/output shapes
    ☐ Check activation functions are appropriate
    ☐ Confirm loss function matches output type
    ☐ Validate optimizer parameters
    ☐ Check for gradient flow (vanishing/exploding)

    TRAINING LOOP:
    ☐ Monitor loss for NaN/Inf values
    ☐ Check learning rate (too high/low?)
    ☐ Verify batch size fits in memory
    ☐ Confirm data augmentation is working
    ☐ Check for data leakage between splits
    ☐ Validate metric calculations

    ENVIRONMENT:
    ☐ Confirm package versions match
    ☐ Check CUDA/GPU availability if expected
    ☐ Verify random seeds for reproducibility
    ☐ Check file paths

<a id='prevention'></a>
## 🛡️ Preventive Practices

The best bug is the one you prevent. Let's explore practices that catch errors early.


In [30]:
# 💡 Type Hints for ML Functions

from typing import Tuple, Union, List
import numpy as np
import numpy.typing as npt

# Without type hints - unclear what types are expected
def train_model_vague(data, labels, epochs):
    pass

# With type hints - clear contract
def train_model_typed(
    data: npt.NDArray[np.float64],
    labels: npt.NDArray[np.int64],
    epochs: int,
    batch_size: int = 32
) -> Tuple[npt.NDArray[np.float64], dict]:
    """
    Train a model with clear type expectations.
    
    Args:
        data: Training features, shape (n_samples, n_features)
        labels: Target values, shape (n_samples,)
        epochs: Number of training epochs
        batch_size: Samples per batch
        
    Returns:
        Tuple of (predictions, training_history)
    """
    # Type hints help IDEs catch errors and document expectations
    assert data.ndim == 2, f"Expected 2D data, got {data.ndim}D"
    assert labels.ndim == 1, f"Expected 1D labels, got {labels.ndim}D"
    assert data.shape[0] == labels.shape[0], "Data/label count mismatch"
    
    # Simulate training
    predictions = np.random.randn(len(labels))
    history = {'loss': [0.5, 0.3, 0.2][:epochs]}
    
    return predictions, history

# Usage with correct types
X = np.random.randn(100, 10).astype(np.float64)
y = np.random.randint(0, 2, 100).astype(np.int64)

preds, hist = train_model_typed(X, y, epochs=3)
print(f"✅ Training completed with typed function")
print(f"   Predictions shape: {preds.shape}")
print(f"   Loss history: {hist['loss']}")

# Type hints also enable static analysis with mypy
print("\n💡 Tip: Use mypy for static type checking: pip install mypy")

✅ Training completed with typed function
   Predictions shape: (100,)
   Loss history: [0.5, 0.3, 0.2]

💡 Tip: Use mypy for static type checking: pip install mypy


In [31]:
# 💡 Input Validation Functions

from typing import Any, Callable
import functools
import numpy as np

def validate_array(
    min_ndim: int = None,
    max_ndim: int = None,
    dtype: type = None,
    min_val: float = None,
    max_val: float = None,
    no_nan: bool = True,
    no_inf: bool = True
):
    """Decorator for array validation."""
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(array: np.ndarray, *args, **kwargs):
            # Type check
            if not isinstance(array, np.ndarray):
                raise TypeError(f"Expected ndarray, got {type(array)}")
            
            # Dimension check
            if min_ndim is not None and array.ndim < min_ndim:
                raise ValueError(f"Expected ndim >= {min_ndim}, got {array.ndim}")
            if max_ndim is not None and array.ndim > max_ndim:
                raise ValueError(f"Expected ndim <= {max_ndim}, got {array.ndim}")
            
            # Dtype check
            if dtype is not None and array.dtype != dtype:
                raise TypeError(f"Expected dtype {dtype}, got {array.dtype}")
            
            # Value range check
            if min_val is not None and array.min() < min_val:
                raise ValueError(f"Values below minimum {min_val}")
            if max_val is not None and array.max() > max_val:
                raise ValueError(f"Values above maximum {max_val}")
            
            # NaN/Inf check
            if no_nan and np.isnan(array).any():
                raise ValueError("Array contains NaN values")
            if no_inf and np.isinf(array).any():
                raise ValueError("Array contains Inf values")
            
            return func(array, *args, **kwargs)
        return wrapper
    return decorator

@validate_array(min_ndim=2, max_ndim=2, no_nan=True, no_inf=True)
def normalize_features(features: np.ndarray) -> np.ndarray:
    """Normalize features with automatic validation."""
    mean = features.mean(axis=0)
    std = features.std(axis=0)
    std[std == 0] = 1  # Prevent division by zero
    return (features - mean) / std

# Valid usage
valid_data = np.random.randn(10, 5)
normalized = normalize_features(valid_data)
print(f"✅ Normalized shape: {normalized.shape}")

# Invalid usage (uncomment to see validation errors):
# invalid_1d = np.random.randn(10)
# normalize_features(invalid_1d)  # ValueError: ndim

# invalid_nan = np.array([1, 2, np.nan, 4])
# normalize_features(invalid_nan)  # ValueError: NaN

print("\n💡 Validation decorators catch errors at function entry!")

✅ Normalized shape: (10, 5)

💡 Validation decorators catch errors at function entry!


In [32]:
# 💡 Logging for ML Pipelines

import logging
import sys
import numpy as np
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('training.log')
    ]
)

logger = logging.getLogger('MLPipeline')

class LoggedTrainer:
    """Trainer with comprehensive logging for debugging."""
    
    def __init__(self, model_config: dict):
        self.config = model_config
        logger.info(f"Initialized trainer with config: {model_config}")
    
    def load_data(self, data_path: str):
        """Load data with logging."""
        logger.info(f"Loading data from {data_path}")
        
        # Simulate loading
        data = np.random.randn(1000, 10)
        labels = np.random.randint(0, 2, 1000)
        
        logger.info(f"Loaded {len(data)} samples with shape {data.shape}")
        logger.debug(f"Data range: [{data.min():.2f}, {data.max():.2f}]")
        
        return data, labels
    
    def train_epoch(self, data, labels, epoch: int):
        """Train one epoch with progress logging."""
        logger.info(f"Starting epoch {epoch}")
        
        batch_size = self.config.get('batch_size', 32)
        n_batches = len(data) // batch_size
        
        losses = []
        for batch_idx in range(n_batches):
            batch_data = data[batch_idx*batch_size:(batch_idx+1)*batch_size]
            batch_labels = labels[batch_idx*batch_size:(batch_idx+1)*batch_size]
            
            # Simulate training step
            loss = np.random.random() * 0.1 + 0.5 * (1 / (epoch + 1))
            losses.append(loss)
            
            # Log every N batches
            if batch_idx % 10 == 0:
                logger.debug(f"Epoch {epoch}, Batch {batch_idx}/{n_batches}, Loss: {loss:.4f}")
            
            # Check for issues
            if np.isnan(loss):
                logger.error(f"NaN loss detected at epoch {epoch}, batch {batch_idx}!")
                raise ValueError("Training diverged")
        
        avg_loss = np.mean(losses)
        logger.info(f"Epoch {epoch} completed. Average loss: {avg_loss:.4f}")
        
        return avg_loss

# Usage
trainer = LoggedTrainer({'batch_size': 32, 'lr': 0.001})
data, labels = trainer.load_data('data/train.csv')

for epoch in range(3):
    loss = trainer.train_epoch(data, labels, epoch)

print("\n💡 Check training.log file for full debug history!")
print("💡 Use logging.DEBUG level for verbose intermediate values")

2026-04-07 06:08:30,619 - MLPipeline - INFO - Initialized trainer with config: {'batch_size': 32, 'lr': 0.001}
2026-04-07 06:08:30,622 - MLPipeline - INFO - Loading data from data/train.csv
2026-04-07 06:08:30,625 - MLPipeline - INFO - Loaded 1000 samples with shape (1000, 10)
2026-04-07 06:08:30,628 - MLPipeline - INFO - Starting epoch 0
2026-04-07 06:08:30,630 - MLPipeline - INFO - Epoch 0 completed. Average loss: 0.5373
2026-04-07 06:08:30,632 - MLPipeline - INFO - Starting epoch 1
2026-04-07 06:08:30,635 - MLPipeline - INFO - Epoch 1 completed. Average loss: 0.2989
2026-04-07 06:08:30,637 - MLPipeline - INFO - Starting epoch 2
2026-04-07 06:08:30,639 - MLPipeline - INFO - Epoch 2 completed. Average loss: 0.2212

💡 Check training.log file for full debug history!
💡 Use logging.DEBUG level for verbose intermediate values


<a id='exercises'></a>
## 🛠️ Hands-On Exercises

Now it's your turn! Work through these exercises to build your debugging skills. Try to solve each one before checking the solutions.


### Exercise 1: Fix the NameError
The following code has a NameError. Identify and fix it.


In [33]:
# Exercise 1: Fix the NameError
def calculate_accuracy(predictions, targets):
    correct = (predictions == targets).sum()
    return correct / total  # Bug: 'total' is not defined

# Test with:
preds = np.array([1, 0, 1, 1, 0])
targs = np.array([1, 0, 0, 1, 0])
# print(calculate_accuracy(preds, targs))

### Exercise 2: Debug the Shape Mismatch
Fix the shape mismatch in this matrix multiplication.


In [34]:
# Exercise 2: Debug the Shape Mismatch
import numpy as np

def linear_layer(x, weights, bias):
    """Should compute x @ weights + bias"""
    return np.dot(x, weights) + bias  # Bug somewhere here

# Test data
x = np.random.randn(32, 784)      # 32 samples, 784 features
w = np.random.randn(256, 784)   # 256 outputs, 784 inputs
b = np.random.randn(256)        # 256 bias terms

# output = linear_layer(x, w, b)
# print(f"Output shape: {output.shape}")  # Should be (32, 256)

### Exercise 3: Handle NaN Propagation
Modify this function to detect and handle NaN values gracefully.


In [35]:
# Exercise 3: Handle NaN Propagation
def compute_gradient(loss_history):
    """Compute gradient of loss for early stopping."""
    return np.gradient(loss_history)

# Test with NaN values
losses = np.array([1.0, 0.8, 0.6, np.nan, 0.4])
# grad = compute_gradient(losses)
# print(f"Gradient: {grad}")  # Should handle NaN gracefully

### Exercise 4: Use %debug Magic
Run this cell to trigger an error, then use `%debug` in a new cell to investigate.


In [36]:
# Exercise 4: Use %debug Magic
def process_batch(batch_data):
    """Process a batch of data."""
    normalized = batch_data / batch_data.sum(axis=1, keepdims=True)
    return normalized ** 2

# This will fail - use %debug after to investigate
bad_batch = np.array([[1, 2], [0, 0]])  # Second row sums to 0!
# result = process_batch(bad_batch)

### Exercise 5: Fix the Type Error
Fix the type error in this data preprocessing function.


In [37]:
# Exercise 5: Fix the Type Error
import pandas as pd

def encode_categorical(data):
    """One-hot encode categorical features."""
    return pd.get_dummies(data)

# Test data with mixed types
df = pd.DataFrame({
    'A': ['cat1', 'cat2', 'cat1'],
    'B': [1, 2, 3]  # Numeric column
})
# encoded = encode_categorical(df)
# print(encoded.dtypes)

### Exercise 6: Debug Index Alignment
Fix the pandas index alignment issue in this code.


In [38]:
# Exercise 6: Debug Index Alignment
import pandas as pd

s1 = pd.Series([1, 2, 3, 4], index=['a', 'b', 'c', 'd'])
s2 = pd.Series([10, 20, 30], index=['a', 'b', 'c'])

def combine_series(a, b):
    """Should return element-wise sum."""
    return a + b  # Bug: index alignment causes NaN

# result = combine_series(s1, s2)
# print(result)

### Exercise 7: Fix the Silent Failure
This function silently returns wrong results. Add proper validation.


In [39]:
# Exercise 7: Fix the Silent Failure
def normalize_data(data):
    """Normalize to [0, 1] range."""
    min_val = data.min()
    max_val = data.max()
    return (data - min_val) / (max_val - min_val)  # Bug when max == min

# Test with edge case
constant_data = np.array([5, 5, 5, 5])
# normalized = normalize_data(constant_data)
# print(f"Result: {normalized}")  # Should handle constant data

### Exercise 8: Debug Memory Issue
Optimize this function to avoid memory issues with large arrays.


In [40]:
# Exercise 8: Debug Memory Issue
def create_feature_matrix(n_samples=100000):
    """Creates large feature matrix - may cause MemoryError."""
    # Current implementation creates intermediate arrays
    features = []
    for i in range(n_samples):
        row = np.random.randn(1000)
        features.append(row ** 2 + np.log(np.abs(row) + 1))
    return np.array(features)

# Try with smaller n first, then optimize
# matrix = create_feature_matrix(1000)
# print(f"Matrix shape: {matrix.shape}")

### Exercise 9: Fix Import and Attribute Errors
Fix the import and attribute errors in this ML pipeline stub.


In [41]:
# Exercise 9: Fix Import and Attribute Errors
# This code has multiple import/attribute issues

# from sklearn.preprocessing import StandardScalar  # Wrong name
# from sklearn.model_selection import train_test_split

def preprocess(X):
    # scaler = StandardScalar()  # Wrong class name
    # return scaler.fit_transform(X)
    pass

# X_train, X_test = train_test_split(X, test_size=0.2)  # Wrong function usage

### Exercise 10: Debug Complete ML Pipeline
This mini ML pipeline has multiple bugs. Find and fix them all.


In [42]:
# Exercise 10: Debug Complete ML Pipeline
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def broken_ml_pipeline():
    """This pipeline has multiple bugs. Find and fix them!"""
    
    # Generate synthetic data
    np.random.seed(42)
    X = np.random.randn(100, 5)
    y = np.random.randint(0, 2, 100)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)  # Bug 1?
    
    # Train model
    model = LogisticRegression()
    model.fit(X_train_scaled, y_train)
    
    # Predict
    predictions = model.predict(X_test_scaled)
    
    # Evaluate
    accuracy = accuracy_score(y_test, predictions)
    print(f"Accuracy: {accuracy:.2f}")
    
    return model, accuracy

# Run to find bugs:
# model, acc = broken_ml_pipeline()

<a id='solutions'></a>
## Solutions (Try to Debug First Before Checking!)

Below are detailed solutions with explanations. Only look after attempting the exercises yourself.


### Solution 1: NameError Fix
The variable `total` was never defined. We need to calculate it from the input length.


In [43]:
# Solution 1
def calculate_accuracy_fixed(predictions, targets):
    """Fixed version with proper variable definition."""
    correct = (predictions == targets).sum()
    total = len(predictions)  # Fixed: define total
    return correct / total

preds = np.array([1, 0, 1, 1, 0])
targs = np.array([1, 0, 0, 1, 0])
accuracy = calculate_accuracy_fixed(preds, targs)
print(f"✅ Accuracy: {accuracy:.2%}")
print(f"   (Correct: 3/5 = 60%)")

✅ Accuracy: 80.00%
   (Correct: 3/5 = 60%)


### Solution 2: Shape Mismatch Fix
The weights matrix needs to be transposed or the dimensions reordered. The operation `x @ weights` requires `weights` to have shape `(784, 256)`, not `(256, 784)`.


In [45]:
# Solution 2
def linear_layer_fixed(x, weights, bias):
    """Fixed matrix multiplication."""
    # Fixed: No need to transpose, weights are already (input_dim, output_dim)
    return np.dot(x, weights) + bias
    
    # Option 2: Define weights with correct shape initially
    # weights should be (784, 256) for x (32, 784) @ weights (784, 256) = (32, 256)

# Test with corrected dimensions
x = np.random.randn(32, 784)
w = np.random.randn(784, 256)  # Fixed: input_dim x output_dim
b = np.random.randn(256)

output = linear_layer_fixed(x, w, b)
print(f"✅ Output shape: {output.shape}")  # Should be (32, 256)
print(f"   Expected: (32, 256)")

✅ Output shape: (32, 256)
   Expected: (32, 256)


### Solution 3: NaN Handling
We need to detect NaN values and either raise an error or handle them gracefully (e.g., by interpolation or returning a safe default).


In [46]:
# Solution 3
def compute_gradient_safe(loss_history):
    """Compute gradient with NaN handling."""
    # Check for NaN
    if np.isnan(loss_history).any():
        print("⚠️  Warning: NaN values detected in loss history")
        
        # Option 1: Raise error
        # raise ValueError("Cannot compute gradient with NaN values")
        
        # Option 2: Interpolate NaN values
        mask = ~np.isnan(loss_history)
        indices = np.arange(len(loss_history))
        loss_history = np.interp(indices, indices[mask], loss_history[mask])
        print("   Interpolated NaN values")
    
    return np.gradient(loss_history)

losses = np.array([1.0, 0.8, 0.6, np.nan, 0.4])
grad = compute_gradient_safe(losses)
print(f"✅ Gradient: {grad}")
print(f"   Successfully handled NaN through interpolation")

⚠️  Warning: NaN values detected in loss history
   Interpolated NaN values
✅ Gradient: [-0.2  -0.2  -0.15 -0.1  -0.1 ]
   Successfully handled NaN through interpolation


### Solution 4: Debug with %debug
The error occurs because the second row sums to 0, causing division by zero. In a new cell after running the error cell, run `%debug` and inspect `batch_data.sum(axis=1)` to see the zero sum.


In [47]:
# Solution 4
def process_batch_fixed(batch_data):
    """Process batch with division by zero protection."""
    row_sums = batch_data.sum(axis=1, keepdims=True)
    
    # Check for zero sums
    if (row_sums == 0).any():
        print("⚠️  Warning: Zero-sum rows detected, adding epsilon")
        row_sums = np.where(row_sums == 0, 1e-10, row_sums)
    
    normalized = batch_data / row_sums
    return normalized ** 2

bad_batch = np.array([[1, 2], [0, 0]])
result = process_batch_fixed(bad_batch)
print(f"✅ Result:\n{result}")
print(f"   No division by zero errors!")

⚠️  Warning: Zero-sum rows detected, adding epsilon
✅ Result:
[[0.11111111 0.44444444]
 [0.         0.        ]]
   No division by zero errors!


### Solution 5: Type Error Fix
`pd.get_dummies()` works on DataFrames, but we need to ensure consistent handling. The issue might be wanting to encode only specific columns or handle numeric columns differently. The code actually works, but let's make it more robust.


In [48]:
# Solution 5
def encode_categorical_fixed(data, columns=None):
    """One-hot encode with better type handling."""
    # Select only object/category columns if none specified
    if columns is None:
        columns = data.select_dtypes(include=['object', 'category']).columns
    
    # Convert to categorical to ensure proper encoding
    data_copy = data.copy()
    for col in columns:
        data_copy[col] = data_copy[col].astype('category')
    
    return pd.get_dummies(data_copy, columns=columns)

df = pd.DataFrame({
    'A': ['cat1', 'cat2', 'cat1'],
    'B': [1, 2, 3]
})

encoded = encode_categorical_fixed(df)
print(f"✅ Encoded dtypes:\n{encoded.dtypes}")
print(f"   Shape: {encoded.shape}")

✅ Encoded dtypes:
B         int64
A_cat1     bool
A_cat2     bool
dtype: object
   Shape: (3, 3)


C:\Users\786\AppData\Local\Temp\ipykernel_10364\2690704311.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columns = data.select_dtypes(include=['object', 'category']).columns


### Solution 6: Index Alignment Fix
The issue is that `s1` has index `['a', 'b', 'c', 'd']` while `s2` has `['a', 'b', 'c']`. When adding, pandas aligns by index, so 'd' gets NaN. We need to align them explicitly or use values.


In [49]:
# Solution 6
def combine_series_fixed(a, b):
    """Combine series with proper alignment."""
    # Option 1: Align to common indices only
    a_aligned, b_aligned = a.align(b, join='inner')
    return a_aligned + b_aligned
    
    # Option 2: Reset indices to positional
    # return a.reset_index(drop=True) + b.reset_index(drop=True)
    
    # Option 3: Reindex b to match a, filling missing with 0
    # return a + b.reindex(a.index, fill_value=0)

s1 = pd.Series([1, 2, 3, 4], index=['a', 'b', 'c', 'd'])
s2 = pd.Series([10, 20, 30], index=['a', 'b', 'c'])

result = combine_series_fixed(s1, s2)
print(f"✅ Result:\n{result}")
print(f"   No NaN values from index misalignment!")

✅ Result:
a    11
b    22
c    33
dtype: int64
   No NaN values from index misalignment!


### Solution 7: Silent Failure Fix
When `max_val == min_val` (constant data), we get division by zero, resulting in NaN or Inf. We need to check for this case.


In [50]:
# Solution 7
def normalize_data_fixed(data):
    """Normalize with zero-variance protection."""
    min_val = data.min()
    max_val = data.max()
    
    # Check for constant data
    if max_val == min_val:
        print("⚠️  Warning: Constant data detected, returning zeros")
        return np.zeros_like(data, dtype=float)
    
    normalized = (data - min_val) / (max_val - min_val)
    
    # Verify result
    assert not np.isnan(normalized).any(), "NaN in normalized data"
    assert not np.isinf(normalized).any(), "Inf in normalized data"
    
    return normalized

constant_data = np.array([5, 5, 5, 5])
normalized = normalize_data_fixed(constant_data)
print(f"✅ Result: {normalized}")
print(f"   Safely handled constant input!")

⚠️  Warning: Constant data detected, returning zeros
✅ Result: [0. 0. 0. 0.]
   Safely handled constant input!


### Solution 8: Memory Optimization
The current implementation creates many intermediate arrays. We should pre-allocate the result array or use generators for very large data.


In [51]:
# Solution 8
def create_feature_matrix_optimized(n_samples=100000):
    """Memory-efficient feature matrix creation."""
    # Pre-allocate array (single memory block)
    features = np.empty((n_samples, 1000), dtype=np.float64)
    
    # Fill in chunks to avoid temporary arrays
    chunk_size = 1000
    for i in range(0, n_samples, chunk_size):
        end = min(i + chunk_size, n_samples)
        chunk = np.random.randn(end - i, 1000)
        
        # Compute features in-place
        features[i:end] = chunk ** 2 + np.log(np.abs(chunk) + 1)
    
    return features

# Test with smaller size first
matrix = create_feature_matrix_optimized(1000)
print(f"✅ Matrix shape: {matrix.shape}")
print(f"   Memory-efficient creation using pre-allocation!")
print(f"   Sample values: {matrix[0, :5]}")

✅ Matrix shape: (1000, 1000)
   Memory-efficient creation using pre-allocation!
   Sample values: [2.85484294 0.22640213 0.18051546 2.40308654 1.7728227 ]


### Solution 9: Import and Attribute Fixes
Multiple issues: `StandardScalar` should be `StandardScaler`, and `train_test_split` usage is incorrect (needs both X and y).


In [52]:
# Solution 9
from sklearn.preprocessing import StandardScaler  # Fixed: Scaler not Scalar
from sklearn.model_selection import train_test_split
import numpy as np

def preprocess_fixed(X):
    """Fixed preprocessing."""
    scaler = StandardScaler()  # Fixed class name
    return scaler.fit_transform(X)

# Generate sample data
X = np.random.randn(100, 5)
y = np.random.randint(0, 2, 100)

# Fixed: train_test_split needs both X and y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_processed = preprocess_fixed(X_train)
print(f"✅ Preprocessed shape: {X_train_processed.shape}")
print(f"   Mean (should be ~0): {X_train_processed.mean():.6f}")
print(f"   Std (should be ~1): {X_train_processed.std():.6f}")

✅ Preprocessed shape: (80, 5)
   Mean (should be ~0): 0.000000
   Std (should be ~1): 1.000000


### Solution 10: Complete Pipeline Debug
The pipeline actually works, but let's add robust error handling and validation to make it production-ready.


In [53]:
# Solution 10
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def robust_ml_pipeline():
    """Production-ready ML pipeline with validation."""
    
    # Generate synthetic data
    np.random.seed(42)
    X = np.random.randn(100, 5)
    y = np.random.randint(0, 2, 100)
    
    # Validate data
    assert not np.isnan(X).any(), "X contains NaN"
    assert not np.isnan(y).any(), "y contains NaN"
    assert len(np.unique(y)) >= 2, "Need at least 2 classes"
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Verify scaling
    assert np.allclose(X_train_scaled.mean(axis=0), 0, atol=1e-7)
    assert np.allclose(X_train_scaled.std(axis=0), 1, atol=1e-7)
    
    # Train model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_scaled, y_train)
    
    # Predict
    predictions = model.predict(X_test_scaled)
    
    # Evaluate
    accuracy = accuracy_score(y_test, predictions)
    print(f"✅ Accuracy: {accuracy:.2%}")
    
    # Additional metrics
    print(f"   Baseline (majority class): {max(np.bincount(y_test)) / len(y_test):.2%}")
    
    return model, accuracy

model, acc = robust_ml_pipeline()
print(f"\n💡 Pipeline completed successfully with validation at each step!")

Train size: 80, Test size: 20
✅ Accuracy: 35.00%
   Baseline (majority class): 55.00%

💡 Pipeline completed successfully with validation at each step!
